In [1]:
import numpy as np
import scipy.io
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

sub = 1
offset = 0
T_in = 1
T = 10

# --- carica ---  #
data   = scipy.io.loadmat('cylinder_wake.mat')
X_star = data['X_star']           # (N,2)
U_star = data['U_star']           # (N,2,T)
P_star = data['p_star']           # (N,T)
t_star = data['t'].squeeze()      # (T,)

# --- griglia regolare (completa) ---
dec =12
x_unique = np.unique(np.round(X_star[:,0], dec))   # 100 valori
y_unique = np.unique(np.round(X_star[:,1], dec))   # 50 valori
Sx_full, Sy_full = len(x_unique), len(y_unique)
time = t_star.shape[0]

# mapping (x,y) -> (i,j) sulla griglia COMPLETA
x_to_i = {x:i for i,x in enumerate(x_unique)}
y_to_j = {y:j for j,y in enumerate(y_unique)}
ij = np.empty((X_star.shape[0],2), dtype=int)
for k,(x,y) in enumerate(np.round(X_star, dec)):
    ij[k] = (x_to_i[x], y_to_j[y])

# --- riempi griglie complete ---
U_grid = np.empty((Sx_full, Sy_full, time, 2), dtype=U_star.dtype)  # ...,(u,v)
P_grid = np.empty((Sx_full, Sy_full, time),     dtype=P_star.dtype)

for k in range(X_star.shape[0]):
    i,j = ij[k]
    U_grid[i,j,:,0] = U_star[k,0,:]   # u
    U_grid[i,j,:,1] = U_star[k,1,:]   # v
    P_grid[i,j,:]   = P_star[k,:]

# ====== QUI scegli metà dominio in x (50 colonne) ======
half = Sx_full // 2

# Metà SINISTRA:    ix = slice(0, half)
# Metà DESTRA:      ix = slice(half, Sx_full)
# Metà CENTRATA:    c = Sx_full//2; w = half; ix = slice(c - w//2, c + w//2)
ix = slice(0, half)

# Integra anche l'eventuale subsampling
iy = slice(0, Sy_full)               # tieni tutte le 50 in y
ix = slice(ix.start, ix.stop, sub)   # es. sub=1 → tutte; sub=2 → una ogni 2
iy = slice(iy.start, iy.stop, sub)

# --- in torch, con B=1: u,v,p hanno shape (1, Sx', Sy', T) ---
u = torch.from_numpy(U_grid[ix, iy, :, 0]).unsqueeze(0).contiguous()   # (1, 50, 50, T) se sub=1
v = torch.from_numpy(U_grid[ix, iy, :, 1]).unsqueeze(0).contiguous()
p = torch.from_numpy(P_grid[ix, iy, :]).unsqueeze(0).contiguous()

# --- coord grid coerenti (usando meshgrid) ---
x_half = x_unique[ix]
y_half = y_unique[iy]
Xv, Yv = np.meshgrid(x_half, y_half, indexing='ij')  # (Sx',Sy')
X = torch.from_numpy(Xv)[None, ..., None]  # (1,Sx',Sy',1)
Y = torch.from_numpy(Yv)[None, ..., None]  # (1,Sx',Sy',1)

# --- split in/stimolo e target ---
u_in = u[..., offset:offset+T_in]                         # (1,Sx',Sy',T_in)
v_in = v[..., offset:offset+T_in]
p_in = p[..., offset:offset+T_in]

u_t = u[..., offset+T_in:offset+T_in+T]                   # (1,Sx',Sy',T)
v_t = v[..., offset+T_in:offset+T_in+T]
p_t = p[..., offset+T_in:offset+T_in+T]

# pack finale (B=1): concateni (x,y) ai canali temporali
train_in = torch.cat([X.to(device), Y.to(device), 
                      u_in.to(device), v_in.to(device), p_in.to(device)], dim=-1)  # (1,Sx',Sy', 2+3*T_in)

target = torch.stack([u_t, v_t, p_t], dim=-1)  # (1,Sx',Sy', T, 3)
train_t = target.reshape(target.shape[0], target.shape[1], target.shape[2], -1).contiguous().to(device)

print("Shapes ->", "train_in:", tuple(train_in.shape), "train_t:", tuple(train_t.shape),
      "| sub =", sub, "| half-x =", (ix.start, ix.stop, ix.step))

train_loader = torch.utils.data.DataLoader(
    torch.utils.data.TensorDataset(train_in, train_t),
    batch_size=1, shuffle=False
)

Shapes -> train_in: (1, 50, 50, 5) train_t: (1, 50, 50, 30) | sub = 1 | half-x = (0, 50, 1)


In [2]:
import torch
import numpy as np
import torch.nn as nn
import torch.nn.functional as F

import matplotlib.pyplot as plt
from utilities3 import *

import operator
from functools import reduce
from functools import partial
import os

from timeit import default_timer
import scipy.io
import pandas as pd
from get_params import get_params

import math


torch.manual_seed(0)
np.random.seed(0)

################################################################
# fourier layer
################################################################

def compl_mul2d_rfft(a, b):
    # a: (batch, in_ch, m1, m2, 2)
    # b: (in_ch, out_ch, m1, m2, 2)
    op = partial(torch.einsum, "bctq,dctq->bdtq")
    real = op(a[...,0], b[...,0]) - op(a[...,1], b[...,1])
    imag = op(a[...,1], b[...,0]) + op(a[...,0], b[...,1])
    return torch.stack([real, imag], dim=-1)


class SpectralConv2d_fast(nn.Module):
    def __init__(self, in_channels, out_channels, modes1, modes2):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.modes1 = modes1
        self.modes2 = modes2
        self.scale = (1/(in_channels*out_channels))
        self.weights1 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, modes1, modes2, 2))
        self.weights2 = nn.Parameter(self.scale * torch.rand(in_channels, out_channels, modes1, modes2, 2))

    def forward(self, x):
        batchsize, _, nx, ny = *x.shape, 
        # Compute Fourier coefficients (real input -> rfft2)
        x_ft_complex = torch.fft.rfft2(x, norm='ortho')
        # Convert to real-imag tensor
        x_ft = torch.stack([x_ft_complex.real, x_ft_complex.imag], dim=-1)
        # Allocate output in Fourier space
        out_ft = torch.zeros(batchsize, self.out_channels, nx, ny//2+1, 2, device=x.device)
        # Multiply modes
        out_ft[:, :, :self.modes1, :self.modes2] = \
            compl_mul2d_rfft(x_ft[:, :, :self.modes1, :self.modes2], self.weights1)
        out_ft[:, :, -self.modes1:, :self.modes2] = \
            compl_mul2d_rfft(x_ft[:, :, -self.modes1:, :self.modes2], self.weights2)
        # Convert back to complex
        out_ft_complex = torch.complex(out_ft[...,0], out_ft[...,1])
        # Return to physical space
        x = torch.fft.irfft2(out_ft_complex, s=(nx, ny), norm='ortho')
        return x

class SimpleBlock2d(nn.Module):
    def __init__(self, modes1, modes2, width):
        super(SimpleBlock2d, self).__init__()

        self.modes1 = modes1
        self.modes2 = modes2
        self.width = width
        self.fc0 = nn.Linear(5, self.width)

        self.conv0 = SpectralConv2d_fast(self.width, self.width, self.modes1, self.modes2)
        self.conv1 = SpectralConv2d_fast(self.width, self.width, self.modes1, self.modes2)
        self.conv2 = SpectralConv2d_fast(self.width, self.width, self.modes1, self.modes2)
        self.conv3 = SpectralConv2d_fast(self.width, self.width, self.modes1, self.modes2)
        self.w0 = nn.Conv1d(self.width, self.width, 1)
        self.w1 = nn.Conv1d(self.width, self.width, 1)
        self.w2 = nn.Conv1d(self.width, self.width, 1)
        self.w3 = nn.Conv1d(self.width, self.width, 1)
        self.bn0 = torch.nn.BatchNorm2d(self.width)
        self.bn1 = torch.nn.BatchNorm2d(self.width)
        self.bn2 = torch.nn.BatchNorm2d(self.width)
        self.bn3 = torch.nn.BatchNorm2d(self.width)


        self.fc1 = nn.Linear(self.width, 128)
        self.fc2 = nn.Linear(128, 3)

    def forward(self, x):
        batchsize = x.shape[0]
        size_x, size_y = x.shape[1], x.shape[2]

        x = self.fc0(x)
        x = x.permute(0, 3, 1, 2)

        x1 = self.conv0(x)
        x2 = self.w0(x.view(batchsize, self.width, -1)).view(batchsize, self.width, size_x, size_y)
        x = self.bn0(x1 + x2)
        x = F.relu(x)
        x1 = self.conv1(x)
        x2 = self.w1(x.view(batchsize, self.width, -1)).view(batchsize, self.width, size_x, size_y)
        x = self.bn1(x1 + x2)
        x = F.relu(x)
        x1 = self.conv2(x)
        x2 = self.w2(x.view(batchsize, self.width, -1)).view(batchsize, self.width, size_x, size_y)
        x = self.bn2(x1 + x2)
        x = F.relu(x)
        x1 = self.conv3(x)
        x2 = self.w3(x.view(batchsize, self.width, -1)).view(batchsize, self.width, size_x, size_y)
        x = self.bn3(x1 + x2)


        x = x.permute(0, 2, 3, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        return x

class Net2d(nn.Module):
    def __init__(self, modes, width):
        super(Net2d, self).__init__()

        self.conv1 = SimpleBlock2d(modes, modes, width)


    def forward(self, x):
        x = self.conv1(x)
        return x


    def count_params(self):
        c = 0
        for p in self.parameters():
            c += reduce(operator.mul, list(p.size()))

        return c

import torch
import math

# ------------------------------------------------------------
# 1) Fourier continuation (even-reflection) 2D
#    Estende con riflessione pari (C^0 e con pendenze ~0 agli estremi).
#    È robusta per derivazione spettrale su domini non periodici.
# ------------------------------------------------------------
def fc_pad2d_even(x, m: int, dimx: int=-2, dimy: int=-1):
    """
    x: (..., Sx, Sy)
    m: spessore continuation per lato
    return: x_ext (..., Sx+2m, Sy+2m)
    """
    assert m > 0
    # pad lungo x (righe)
    x_left  = torch.flip(x[..., 1:m+1, :], dims=(dimx,))   # riflessione interna (evita duplicare bordo)
    x_right = torch.flip(x[..., -m-1:-1, :], dims=(dimx,))
    x_px = torch.cat([x_left, x, x_right], dim=dimx)
    # pad lungo y (colonne)
    y_top    = torch.flip(x_px[..., :, 1:m+1], dims=(dimy,))
    y_bottom = torch.flip(x_px[..., :, -m-1:-1], dims=(dimy,))
    x_ext = torch.cat([y_top, x_px, y_bottom], dim=dimy)
    return x_ext

def center_crop2d(x_ext, m: int, dimx: int=-2, dimy: int=-1):
    """
    x_ext: (..., Sx+2m, Sy+2m)
    return: (..., Sx, Sy)
    """
    slx = [slice(None)] * x_ext.ndim
    sly = [slice(None)] * x_ext.ndim
    Sx_ext = x_ext.shape[dimx]
    Sy_ext = x_ext.shape[dimy]
    Sx = Sx_ext - 2*m
    Sy = Sy_ext - 2*m
    slx[dimx] = slice(m, m+Sx)
    sly[dimy] = slice(m, m+Sy)
    return x_ext[tuple(slx)][tuple(sly)]

# ------------------------------------------------------------
# 2) Wavenumbers e operatori spettrali su griglia estesa
# ------------------------------------------------------------
def _build_kxy(Sx, Sy, device):
    """
    Ritorna kx, ky (float) di shape (Sx, Sy) per FFT periodica su griglia Sx,Sy.
    """
    kx = torch.fft.fftfreq(Sx, d=1.0, device=device)  # [0,1/Sx,...,-1/Sx]
    ky = torch.fft.fftfreq(Sy, d=1.0, device=device)
    kx = kx.view(Sx, 1).expand(Sx, Sy)
    ky = ky.view(1, Sy).expand(Sx, Sy)
    # fattori 2π
    kx = (2.0*math.pi) * kx
    ky = (2.0*math.pi) * ky
    return kx, ky

def _dealias_23(mask_shape, device):
    """
    Maschera 2/3 classica per antialiasing (sfera rettangolare).
    """
    Sx, Sy = mask_shape
    cx = Sx//2
    cy = Sy//2
    kx = torch.fft.fftfreq(Sx, d=1.0, device=device).abs()
    ky = torch.fft.fftfreq(Sy, d=1.0, device=device).abs()
    # cutoff = 1/3 in frequenza normale (⇒ 2/3 dei modes tenuti)
    cutoff_x = (1.0/3.0)
    cutoff_y = (1.0/3.0)
    mx = (kx <= cutoff_x).view(Sx, 1).expand(Sx, Sy)
    my = (ky <= cutoff_y).view(1, Sy).expand(Sx, Sy)
    return (mx & my)

def _spectral_derivatives_on_extended(f_ext, dealias=True):
    """
    f_ext: (B, SxE, SyE) campo scalare su griglia ESTESA (già periodica per costruzione)
    Ritorna: fx_ext, fy_ext, lap_ext (stesse shape)
    """
    B, SxE, SyE = f_ext.shape
    device = f_ext.device

    kx, ky = _build_kxy(SxE, SyE, device)               # (SxE, SyE)
    k2 = kx**2 + ky**2
    k2[0,0] = 1.0  # evita div/0 quando serve

    F = torch.fft.fftn(f_ext, dim=(-2, -1))             # (B, SxE, SyE)
    if dealias:
        mask = _dealias_23((SxE, SyE), device)          # bool
        F = F * mask                                   # broadcasting su B

    Fx = (1j * kx) * F
    Fy = (1j * ky) * F
    Flap = -(k2) * F

    fx_ext  = torch.fft.ifftn(Fx,   dim=(-2, -1)).real
    fy_ext  = torch.fft.ifftn(Fy,   dim=(-2, -1)).real
    lap_ext = torch.fft.ifftn(Flap, dim=(-2, -1)).real
    return fx_ext, fy_ext, lap_ext

# ------------------------------------------------------------
# 3) Operatori differenziali per (u,v,p) con FC automatica
# ------------------------------------------------------------
def spectral_ops_vp_fc(u, v, p, m: int=12, dealias: bool=True):
    """
    Calcola operatori differenziali per Navier-Stokes in forma (u,v,p)
    usando Fourier continuation (even-reflection) e derivate spettrali.

    Input:
      u,v,p: (B, Sx, Sy)  tensors
      m: spessore banda FC (celle per lato)
      dealias: applica o meno maschera 2/3 in Fourier

    Output: dict con
      ux, uy, vx, vy,         # derivate prime
      lap_u, lap_v,           # Laplaciani
      div_u,                  # divergenza
      px, py,                 # gradiente pressione
      convx, convy,           # (u·∇)u componenti
      interior_mask           # (1,Sx,Sy) True nell'interno (esclude banda FC)
    """
    assert u.shape == v.shape == p.shape
    B, Sx, Sy = u.shape
    device = u.device

    # 3.1 FC padding per ciascun canale
    u_ext = fc_pad2d_even(u, m)   # (B, Sx+2m, Sy+2m)
    v_ext = fc_pad2d_even(v, m)
    p_ext = fc_pad2d_even(p, m)

    # 3.2 Derivate spettrali su esteso
    ux_ext, uy_ext, lap_u_ext = _spectral_derivatives_on_extended(u_ext, dealias=dealias)
    vx_ext, vy_ext, lap_v_ext = _spectral_derivatives_on_extended(v_ext, dealias=dealias)
    px_ext, py_ext, _         = _spectral_derivatives_on_extended(p_ext, dealias=dealias)

    # 3.3 Crop centrale (tornare alla griglia originale)
    ux   = center_crop2d(ux_ext,   m)
    uy   = center_crop2d(uy_ext,   m)
    vx   = center_crop2d(vx_ext,   m)
    vy   = center_crop2d(vy_ext,   m)
    lap_u= center_crop2d(lap_u_ext,m)
    lap_v= center_crop2d(lap_v_ext,m)
    px   = center_crop2d(px_ext,   m)
    py   = center_crop2d(py_ext,   m)

    # 3.4 Divergenza e convettivo (sulla griglia originale)
    div_u = ux + vy
    convx = u * ux + v * uy
    convy = u * vx + v * vy

    # 3.5 Maschera interno (True solo lontano dai bordi fisici)
    interior_mask = torch.zeros(1, Sx, Sy, dtype=torch.bool, device=device)
    interior_mask[:, m:Sx-m, m:Sy-m] = True

    return {
        'ux': ux, 'uy': uy, 'vx': vx, 'vy': vy,
        'lap_u': lap_u, 'lap_v': lap_v,
        'div_u': div_u,
        'px': px, 'py': py,
        'convx': convx, 'convy': convy,
        'interior_mask': interior_mask
    }


In [3]:
################################################################
# training and evaluation
################################################################
# Parametri\ 

ntrain      = 1

modes       = 24
width       = 64


batch_size  = 1
epochs      = 100

learning_rate   = 2e-4
scheduler_step  = 20
scheduler_gamma = 0.5


step        = 1

rf          = 8
orders_v    = [2,2]
n_samples   = 4
model = Net2d(modes, width).cuda()
# model = torch.load('model/ns_fourier_V100_N1000_ep100_m8_w10')

print(model.count_params())
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=scheduler_step, gamma=scheduler_gamma)


myloss = LpLoss(size_average=False)

test_full = []

path = f'FNO_Multi_Stage_ep{epochs}_m{modes}_w{width}_T{T}'

path_model     = os.path.join('model',  path)
path_image     = os.path.join('image',  path)
path_loss_dir  = os.path.join('loss',   path)
os.makedirs(path_image, exist_ok=True)
os.makedirs(path_loss_dir, exist_ok=True)
os.makedirs(os.path.dirname(path_model), exist_ok=True)
# ============================
# Training loop robusto a T lunghi
# ============================

import numpy as np
import torch

# ---- impostazioni autoregressione / stabilità ----
predict_delta   = False     # True: modello predice Δ; False: predice stato assoluto
use_detach      = True     # stacca il gradiente quando reinserisci la predizione in finestra
time_weighting  = True     # pesa di più gli step temporali lontani
noise_std       = 0.0      # es. 0.01 se lavori con canali normalizzati (std~1). 0.0 = off
grad_clip       = 1.0      # clipping del gradiente (None o 0.0 per disattivare)

# ---- scheduled sampling con "pavimento" ----
tf_max = 1.0
tf_min = 0.10                   # non scendere mai sotto il 10% di teacher forcing
warm   = int(0.6 * 200)      # epoche su cui far decrescere p GT -> tf_min

def get_tf_ratio(ep):
    # Cosine decay da 1.0 a tf_min durante il warmup
    x = min(1.0, ep / max(1, warm))
    # 0 -> 1: cos decresce da 1 a -1
    return float(tf_min + 0.5*(tf_max - tf_min)*(1.0 + torch.cos(torch.tensor(np.pi*x)).item()))

def make_boundary_mask(Sx, Sy, thickness=1, device='cuda'):
    bm = torch.zeros(1, Sx, Sy, 1, device=device)
    t = int(thickness)
    bm[:, :t, :, :]  = 1;  bm[:, -t:, :, :] = 1
    bm[:, :, :t, :]  = 1;  bm[:, :, -t:, :] = 1
    return bm  # shape (1,Sx,Sy,1)

C = 3  # (u,v,p) per passo


37774979


In [4]:
nu = 0.01
dt = 0.1

lambda_res = 1.0      # peso della residual loss (tuning a piacere)
beta_div   = 1.0      # peso del vincolo di incomprimibilità
m_fc       = 12       # spessore banda Fourier continuation

C = 3
# Pesi per fase (modifica liberamente)
PHASES = [
    {
        'name': 'P1_border_only',
        'epochs':200,
        'lambda_bd': 1.0,
        'lambda_res': 0.0,
        'record': False,
        'learning_rate': 1e-3,
        'scheduler_gamma' : 0.5

    },
     {
         'name': 'P2_res_warm',
         'epochs': 200,
         'lambda_bd': 1.0,
         'lambda_res': 0.2,
         'record': True,
         'learning_rate': 3e-5,
         'scheduler_gamma' : 0.01
    }
]

for iph, phase in enumerate(PHASES, 1):

    learning_rate = phase['learning_rate']
    scheduler_gamma = phase['scheduler_gamma']

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)

    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=phase['scheduler_gamma'])


    lambda_bd  = phase['lambda_bd']
    lambda_res = phase['lambda_res']

    print(f"\n===== Phase {iph}/{len(PHASES)}: {phase['name']}  (epochs={phase['epochs']}) =====")
    print(f"lambdas → bd:{lambda_bd}  res:{lambda_res}")

    boundary_loss, full_loss, res_loss = [], [], []

    # usa le epoche della fase
    for ep in range(phase['epochs']):
        model.train()
        t_start = default_timer()

        # accumulatori epoca
        train_l2_step = 0.0           # loss totale per step (tutte le componenti)
        train_l2_full = 0.0           # L2 full-sequence (monitoring)
        train_bd_step = 0.0           # supervisione bordo (u,v)
        train_p_step  = 0.0           # supervisione pressione full
        train_res_step = 0.0          # residual loss

        tf_ratio = get_tf_ratio(ep)

        for xx, yy in train_loader:
            xx = xx.to(device, dtype=torch.float32)  # (B,Sx,Sy, 2 + 3*T_in)
            yy = yy.to(device, dtype=torch.float32)  # (B,Sx,Sy, 3*T)

            B, Sx, Sy = xx.shape[0], xx.shape[1], xx.shape[2]
            bd_mask = make_boundary_mask(Sx, Sy, thickness=1, device=xx.device)  # (1,Sx,Sy,1)

            loss = 0.0
            preds = []

            # accumulatori per QUESTO batch (sommati sui T step)
            sum_bd_batch = 0.0
            sum_p_batch  = 0.0
            sum_res_batch = 0.0

            for t in range(T):
                # target (u_{t+1}, v_{t+1}, p_{t+1})
                y  = yy[..., C*t : C*(t+1)]                          # (B,Sx,Sy,3)

                # predizione
                if predict_delta:
                    base = xx[..., -C:]                              # (u_t,v_t,p_t)
                    delta = model(xx)                                # (Δu,Δv,Δp)
                    im = base + delta                                # (u_{t+1}, v_{t+1}, p_{t+1})
                else:
                    im = model(xx)

                preds.append(im)

                # ==== stati t e t+1 ====
                state_t = xx[..., -C:]         # (B,Sx,Sy,3)
                u_t, v_t, p_t = state_t[...,0], state_t[...,1], state_t[...,2]
                u_n, v_n, p_n = im[...,0],      im[...,1],      im[...,2]

                # Gauge pressione: rimuovi media per stabilità (evita drift)
                def zero_mean(x): 
                    return x - x.mean(dim=(-2,-1), keepdim=True)
                p_t = zero_mean(p_t)
                p_n = zero_mean(p_n)

                # ==== deriv. spettrali con FC su t e t+1 (più robuste) ====
                m_used = 1   # m_fc=8–12 raccomandato
                ops_t = spectral_ops_vp_fc(u_t, v_t, p_t, m=m_used, dealias=True)
                ops_n = spectral_ops_vp_fc(u_n, v_n, p_n, m=m_used, dealias=True)

                # medie stile Crank–Nicolson
                u_bar = 0.5*(u_t + u_n);  v_bar = 0.5*(v_t + v_n)
                ux_b  = 0.5*(ops_t['ux']    + ops_n['ux']);   uy_b  = 0.5*(ops_t['uy']   + ops_n['uy'])
                vx_b  = 0.5*(ops_t['vx']    + ops_n['vx']);   vy_b  = 0.5*(ops_t['vy']   + ops_n['vy'])
                lapu_b= 0.5*(ops_t['lap_u'] + ops_n['lap_u']);lapv_b= 0.5*(ops_t['lap_v']+ ops_n['lap_v'])
                px_b  = 0.5*(ops_t['px']    + ops_n['px']);   py_b  = 0.5*(ops_t['py']   + ops_n['py'])
                div_b = 0.5*(ops_t['div_u'] + ops_n['div_u'])   # o, più severo: ops_n['div_u']

                # ∂/∂t
                dudt = (u_n - u_t) / dt
                dvdt = (v_n - v_t) / dt

                # convettivi medi
                convx_b = u_bar*ux_b + v_bar*uy_b
                convy_b = u_bar*vx_b + v_bar*vy_b

                # residui (f=0): ut + (u·∇)u + ∇p - νΔu = 0
                rx = dudt + convx_b + px_b - nu*lapu_b
                ry = dvdt + convy_b + py_b - nu*lapv_b
                rinc = div_b

                # maschera interno: intersezione (t ∧ t+1)
                mask_int = (ops_t['interior_mask'] & ops_n['interior_mask']).float().expand(B, -1, -1)

                def masked_mse(field, mask):
                    num = ((field**2) * mask).sum()
                    den = mask.sum().clamp_min(1.0)
                    return num / den

                loss_res_t = masked_mse(rx, mask_int) + masked_mse(ry, mask_int) + beta_div*masked_mse(rinc, mask_int)


                # ====== SUPERVISIONE DATI ======
                # u,v SOLO al bordo
                im_vel_bd = im[..., :2] * bd_mask              # (B,Sx,Sy,2)
                y_vel_bd  = y[...,  :2] * bd_mask
                loss_bd_t = myloss(im_vel_bd.reshape(B, -1), y_vel_bd.reshape(B, -1))

                # p su TUTTO il dominio
                im_p = im[..., 2]                               # (B,Sx,Sy)
                y_p  = y[...,  2]
                loss_p_t = myloss(im_p.reshape(B, -1), y_p.reshape(B, -1))

                # somma alla loss totale con i pesi di fase
                loss = loss + lambda_res * loss_res_t + lambda_bd * (loss_bd_t + loss_p_t)

                # ====== scheduled sampling / feed finestra ======
                if torch.rand(1, device=device) < tf_ratio:
                    feed = y
                else:
                    feed = im.detach() if use_detach else im

                if noise_std and noise_std > 0.0:
                    feed = feed + noise_std*torch.randn_like(feed)

                xy   = xx[..., :2]
                hist = xx[..., 2:]
                hist = torch.cat([hist[..., C:], feed], dim=-1)   # sposta la finestra e aggiunge feed
                xx   = torch.cat([xy, hist], dim=-1)

                # ---- accumula per-step (batch) ----
                sum_bd_batch  += loss_bd_t.item()
                sum_p_batch   += loss_p_t.item()
                sum_res_batch += loss_res_t.item()

            # ---- metriche full-sequence ----
            pred_seq = torch.cat(preds, dim=-1)                        # (B,Sx,Sy, 3*T)
            train_l2_full += myloss(pred_seq.reshape(B, -1), yy.reshape(B, -1)).item()

            # ---- accumula su epoca ----
            train_l2_step += loss.item()
            train_bd_step += sum_bd_batch
            train_p_step  += sum_p_batch
            train_res_step += sum_res_batch

            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            if grad_clip and grad_clip > 0.0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        scheduler.step()
        elapsed = default_timer() - t_start
        denom_steps = T * len(train_loader)

        print(f"[Phase {iph}:{phase['name']}] Ep {ep+1:3d}/{phase['epochs']} │ "
              f"Time: {elapsed:6.2f}s │ "
              f"Train step BD(u,v): {train_bd_step/denom_steps:8.4e} │ "
              f"Train step P(full): {train_p_step/denom_steps:8.4e} │ "
              f"Train step RES: {train_res_step/denom_steps:8.4e} │ "
              f"Train step TOTAL: {train_l2_step/denom_steps:8.4e} │ "
              f"Train full L2: {train_l2_full/len(train_loader):8.4e} │ ")

        # salva serie per plotting
        boundary_loss.append(train_bd_step/denom_steps)                     # solo (u,v) bordo
        full_loss.append(train_l2_full/len(train_loader))
        if phase['record']:
            res_loss.append(train_res_step/denom_steps)

    

    idx = list(range(1, phase['epochs']+1))
    data = {
        'loss_boundary':   boundary_loss,
        'loss_full': full_loss,
    }
    if phase['record']:
        data['loss_res'] = res_loss
    df = pd.DataFrame(data, index=idx)
    df.index.name = 'epoch_in_phase'
    csv_path = os.path.join(path_loss_dir, f"loss_{phase['name']}.csv")
    df.to_csv(csv_path)
    print(f"→ Saved loss history for phase '{phase['name']}' in {csv_path}")



===== Phase 1/2: P1_border_only  (epochs=200) =====
lambdas → bd:1.0  res:0.0
[Phase 1:P1_border_only] Ep   1/200 │ Time:   0.61s │ Train step BD(u,v): 1.0225e+00 │ Train step P(full): 1.3251e+00 │ Train step RES: 1.0129e+02 │ Train step TOTAL: 2.3476e+00 │ Train full L2: 1.0763e+00 │ 
[Phase 1:P1_border_only] Ep   2/200 │ Time:   0.25s │ Train step BD(u,v): 6.0181e-01 │ Train step P(full): 1.9056e+00 │ Train step RES: 9.0991e+01 │ Train step TOTAL: 2.5074e+00 │ Train full L2: 1.0261e+00 │ 
[Phase 1:P1_border_only] Ep   3/200 │ Time:   0.27s │ Train step BD(u,v): 2.6070e-01 │ Train step P(full): 5.7203e-01 │ Train step RES: 8.1726e+01 │ Train step TOTAL: 8.3273e-01 │ Train full L2: 9.1187e-01 │ 
[Phase 1:P1_border_only] Ep   4/200 │ Time:   0.28s │ Train step BD(u,v): 2.0920e-01 │ Train step P(full): 7.0475e-01 │ Train step RES: 7.6397e+01 │ Train step TOTAL: 9.1395e-01 │ Train full L2: 8.8348e-01 │ 
[Phase 1:P1_border_only] Ep   5/200 │ Time:   0.27s │ Train step BD(u,v): 1.7838e-01 

In [5]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import numpy as np

# -----------------------------------------
# genera predizioni autoregressive dal modello
# -----------------------------------------
def predict_sequence(model, xx, yy, T, step=1, C=3):
    model.eval()
    preds = []
    with torch.no_grad():
        for t in range(0, T, step):
            y = yy[..., 3*t:3*(t+1)] 
           # ---- predizione ----
            if predict_delta:
                # base = ultimo stato nella finestra (u_t, v_t, p_t)
                base = xx[..., -C:]                                # (B,Sx,Sy,3)
                delta = model(xx)                                  # (B,Sx,Sy,3)
                im = base + delta                                  # pred stato t+1
            else:
                im = model(xx)                                     # pred stato t+1 diretto
            
            preds.append(im)
            
            # aggiorna input con autoregressione
            xy   = xx[..., :2]
            hist = xx[..., 2:]
            hist = torch.cat([hist[..., C:], y], dim=-1)
            xx   = torch.cat([xy, hist], dim=-1)
    pred_seq = torch.cat(preds, dim=-1)  # (B, Sx, Sy, 3*T)
    return pred_seq


# -----------------------------------------
# funzione per animare u,v,p
# -----------------------------------------
def animate_prediction(xx, yy, model, T, step=1, interval=200,
                       save_prefix="anim",  # solo basename del file (niente path)
                       cmap_field="jet", cmap_err="magma",
                       add_colorbars=True, times=None,
                       out_dir=".", save_last_png=True, filename_base=None):
    """
    out_dir: cartella dove salvare i file (es. path_image)
    save_last_png: se True salva anche l'ultimo frame come PNG
    filename_base: basename per i file; se None usa save_prefix
    """
    import os
    if filename_base is None:
        filename_base = save_prefix
    os.makedirs(out_dir, exist_ok=True)

    C = 3
    pred_seq = predict_sequence(model, xx.clone(), yy.clone(), T, step, C)  # (B,Sx,Sy,3*Tpred)
    B, Sx, Sy, _ = pred_seq.shape
    Tpred = pred_seq.shape[-1] // C

    # estrai pred e gt (B,Sx,Sy,T)
    u_pred = pred_seq[..., 0::C]; v_pred = pred_seq[..., 1::C]; p_pred = pred_seq[..., 2::C]
    u_true = yy      [..., 0::C]; v_true = yy      [..., 1::C]; p_true = yy      [..., 2::C]

    # to numpy (Sx,Sy,T)
    u_pred = u_pred.squeeze(0).detach().cpu().numpy()
    v_pred = v_pred.squeeze(0).detach().cpu().numpy()
    p_pred = p_pred.squeeze(0).detach().cpu().numpy()
    u_true = u_true.squeeze(0).detach().cpu().numpy()
    v_true = v_true.squeeze(0).detach().cpu().numpy()
    p_true = p_true.squeeze(0).detach().cpu().numpy()

    # errori (Sx,Sy,T)
    eu = np.abs(u_pred - u_true)
    ev = np.abs(v_pred - v_true)
    ep = np.abs(p_pred - p_true)

    # range colori stabile (dai GT)
    vmin_u, vmax_u = float(u_true.min()), float(u_true.max())
    vmin_v, vmax_v = float(v_true.min()), float(v_true.max())
    vmin_p, vmax_p = float(p_true.min()), float(p_true.max())

    # range errore robusto (99° percentile)
    emax_u = max(1e-12, float(np.percentile(eu, 99)))
    emax_v = max(1e-12, float(np.percentile(ev, 99)))
    emax_p = max(1e-12, float(np.percentile(ep, 99)))

    # figura animazione 3x3
    fig, axes = plt.subplots(3, 3, figsize=(12, 10), constrained_layout=True)
    titles_top = ['u pred', 'v pred', 'p pred']
    titles_mid = ['u GT',   'v GT',   'p GT'  ]
    titles_bot = ['|u err|','|v err|','|p err|']

    for i in range(3):
        axes[0,i].set_title(titles_top[i])
        axes[1,i].set_title(titles_mid[i])
        axes[2,i].set_title(titles_bot[i])
        for r in range(3):
            axes[r,i].set_xticks([]); axes[r,i].set_yticks([])

    k0 = 0
    im00 = axes[0,0].imshow(u_pred[...,k0].T, origin='lower', cmap=cmap_field, vmin=vmin_u, vmax=vmax_u, animated=True)
    im01 = axes[0,1].imshow(v_pred[...,k0].T, origin='lower', cmap=cmap_field, vmin=vmin_v, vmax=vmax_v, animated=True)
    im02 = axes[0,2].imshow(p_pred[...,k0].T, origin='lower', cmap=cmap_field, vmin=vmin_p, vmax=vmax_p, animated=True)

    im10 = axes[1,0].imshow(u_true[...,k0].T, origin='lower', cmap=cmap_field, vmin=vmin_u, vmax=vmax_u, animated=True)
    im11 = axes[1,1].imshow(v_true[...,k0].T, origin='lower', cmap=cmap_field, vmin=vmin_v, vmax=vmax_v, animated=True)
    im12 = axes[1,2].imshow(p_true[...,k0].T, origin='lower', cmap=cmap_field, vmin=vmin_p, vmax=vmax_p, animated=True)

    im20 = axes[2,0].imshow(eu[...,k0].T, origin='lower', cmap=cmap_err,   vmin=0.0, vmax=emax_u, animated=True)
    im21 = axes[2,1].imshow(ev[...,k0].T, origin='lower', cmap=cmap_err,   vmin=0.0, vmax=emax_v, animated=True)
    im22 = axes[2,2].imshow(ep[...,k0].T, origin='lower', cmap=cmap_err,   vmin=0.0, vmax=emax_p, animated=True)

    ims = [im00, im01, im02, im10, im11, im12, im20, im21, im22]
    cbs = []
    if add_colorbars:
        for ax, im in zip(axes[0], ims[0:3]): cbs.append(fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02))
        for ax, im in zip(axes[1], ims[3:6]): cbs.append(fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02))
        for ax, im in zip(axes[2], ims[6:9]): cbs.append(fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02))

    # testo tempo/frame
    if times is not None and len(times) >= Tpred:
        time_text = fig.text(0.01, 0.98, f"t = {float(times[0]):.3f}", fontsize=12)
    else:
        time_text = fig.text(0.01, 0.98, f"frame 0 / {Tpred-1}", fontsize=12)

    def update(frame):
        im00.set_array(u_pred[...,frame].T); im01.set_array(v_pred[...,frame].T); im02.set_array(p_pred[...,frame].T)
        im10.set_array(u_true[...,frame].T); im11.set_array(v_true[...,frame].T); im12.set_array(p_true[...,frame].T)
        im20.set_array(eu[...,frame].T);     im21.set_array(ev[...,frame].T);     im22.set_array(ep[...,frame].T)
        if times is not None and len(times) >= Tpred:
            time_text.set_text(f"t = {float(times[frame]):.3f}")
        else:
            time_text.set_text(f"frame {frame} / {Tpred-1}")
        return ims + [time_text]

    ani = animation.FuncAnimation(fig, update, frames=Tpred, interval=interval, blit=False)

    # ---- salva GIF ----
    gif_path = os.path.join(out_dir, f"{filename_base}.gif")
    ani.save(gif_path, writer='pillow', fps=max(1, int(1000/interval)))

    # ---- salva PNG dell'ultimo frame (stessa griglia 3x3) ----
    png_path = None
    if save_last_png:
        frame = Tpred-1
        fig_last, axes_last = plt.subplots(3, 3, figsize=(12, 10), constrained_layout=True)
        titles_top = ['u pred', 'v pred', 'p pred']
        titles_mid = ['u GT',   'v GT',   'p GT'  ]
        titles_bot = ['|u err|','|v err|','|p err|']
        for i in range(3):
            axes_last[0,i].set_title(titles_top[i])
            axes_last[1,i].set_title(titles_mid[i])
            axes_last[2,i].set_title(titles_bot[i])
            for r in range(3):
                axes_last[r,i].set_xticks([]); axes_last[r,i].set_yticks([])

        axes_last[0,0].imshow(u_pred[...,frame].T, origin='lower', cmap=cmap_field, vmin=vmin_u, vmax=vmax_u)
        axes_last[0,1].imshow(v_pred[...,frame].T, origin='lower', cmap=cmap_field, vmin=vmin_v, vmax=vmax_v)
        axes_last[0,2].imshow(p_pred[...,frame].T, origin='lower', cmap=cmap_field, vmin=vmin_p, vmax=vmax_p)

        axes_last[1,0].imshow(u_true[...,frame].T, origin='lower', cmap=cmap_field, vmin=vmin_u, vmax=vmax_u)
        axes_last[1,1].imshow(v_true[...,frame].T, origin='lower', cmap=cmap_field, vmin=vmin_v, vmax=vmax_v)
        axes_last[1,2].imshow(p_true[...,frame].T, origin='lower', cmap=cmap_field, vmin=vmin_p, vmax=vmax_p)

        axes_last[2,0].imshow(eu[...,frame].T, origin='lower', cmap=cmap_err, vmin=0.0, vmax=emax_u)
        axes_last[2,1].imshow(ev[...,frame].T, origin='lower', cmap=cmap_err, vmin=0.0, vmax=emax_v)
        axes_last[2,2].imshow(ep[...,frame].T, origin='lower', cmap=cmap_err, vmin=0.0, vmax=emax_p)

        if add_colorbars:
            for ax in axes_last[0]: fig_last.colorbar(ax.images[0], ax=ax, fraction=0.046, pad=0.02)
            for ax in axes_last[1]: fig_last.colorbar(ax.images[0], ax=ax, fraction=0.046, pad=0.02)
            for ax in axes_last[2]: fig_last.colorbar(ax.images[0], ax=ax, fraction=0.046, pad=0.02)

        png_path = os.path.join(out_dir, f"{filename_base}_last.png")
        fig_last.savefig(png_path, dpi=160, bbox_inches="tight")
        plt.close(fig_last)

    plt.close(fig)
    return ani, gif_path, png_path



# batch di esempio
xx, yy = next(iter(train_loader))
xx = xx.to(device, dtype=torch.float32)
yy = yy.to(device, dtype=torch.float32)

# salva GIF + ultimo frame PNG nella cartella immagini dell'esperimento
ani, gif_path, png_path = animate_prediction(
    xx, yy, model,
    T=T, step=1, interval=200,
    save_prefix="supervised_fno",
    out_dir=path_image,          # <--- usa la cartella 'image/<path>'
    save_last_png=True,
)

print("GIF salvata in:", gif_path)
print("PNG ultimo frame in:", png_path)




GIF salvata in: image/FNO_Multi_Stage_ep100_m24_w64_T10/supervised_fno.gif
PNG ultimo frame in: image/FNO_Multi_Stage_ep100_m24_w64_T10/supervised_fno_last.png
